# LexAI — Notebook 05: Multimodal Evidence Analysis

Demonstrates analysis of non-text evidence:
- **Image forensics** with GPT-4V (EXIF, tampering, strength scoring)
- **Audio testimony** transcription with OpenAI Whisper
- **Cross-witness inconsistency** detection
- **Evidence matrix** — ranking evidence by strength

Note: Vision/Whisper calls use mock responses when API keys are absent.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
CASES_DIR = pathlib.Path('../data/cases/sample_cases')
print('LexAI Multimodal Evidence Demo')

## 1. Image Evidence Analysis (GPT-4V)

In [ ]:
from src.image_analyzer import LegalImageAnalyzer

analyzer = LegalImageAnalyzer()
print('Image analyzer initialized')
print('Using API:', 'GPT-4V' if analyzer.vision_client.client else 'Mock (no OPENAI_API_KEY)')

In [ ]:
image_path = CASES_DIR / 'case_002' / 'fraud_evidence_photo.jpg'

if image_path.exists():
    result = analyzer.analyze_evidence_image(str(image_path), evidence_type='fraud_evidence')
    print('=== IMAGE ANALYSIS RESULT ===')
    print(f'File: {result.get("file_name")}')
    print(f'Evidence Strength: {result.get("evidence_strength", 0):.1f}/10')
    print(f'Tampering Detected: {result.get("tampering_detected", False)}')
    print(f'\nForensic Analysis:')
    print(result.get('forensic_analysis', '')[:500])
else:
    print('Sample image not found. Run: python -X utf8 scripts/generate_sample_cases.py')

## 2. EXIF Metadata Extraction

In [ ]:
from PIL import Image
from PIL.ExifTags import TAGS
import pathlib

def extract_exif(image_path):
    try:
        img = Image.open(image_path)
        exif_data = img._getexif() or {}
        decoded = {TAGS.get(k, k): str(v)[:60] for k, v in exif_data.items()}
        return decoded
    except Exception as e:
        return {'error': str(e), 'format': Image.open(image_path).format if pathlib.Path(image_path).exists() else 'N/A'}

if image_path.exists():
    exif = extract_exif(image_path)
    print('EXIF Metadata:')
    for k, v in exif.items():
        print(f'  {k}: {v}')
    if not exif or 'error' in exif:
        print('  (No EXIF data — synthetic test image)')

## 3. Audio Testimony Transcription (Whisper)

In [ ]:
from src.audio_transcriber import TestimonyTranscriber

transcriber = TestimonyTranscriber()
print('Transcriber initialized')
print('Whisper available:', transcriber.whisper_available)

audio_path = CASES_DIR / 'case_002' / 'victim_testimony.mp3'
if not audio_path.exists():
    audio_path = CASES_DIR / 'case_001' / 'witness_testimony.wav'

if audio_path.exists():
    result = transcriber.transcribe_testimony(str(audio_path))
    print(f'\nFile: {result.get("file_name")}')
    print(f'Duration: {result.get("duration_seconds", 0):.1f}s')
    print(f'\nTranscript:')
    print(result.get('transcript', '')[:400])
else:
    print('No audio files found in sample cases. Run generate_sample_cases.py')

## 4. Mock Testimony Analysis

In [ ]:
mock_testimony = {
    'file_name': 'victim_testimony.mp3',
    'transcript': 'I met the accused at a conference in January 2024. He showed me a brochure promising 30% returns in 6 months. I transferred 45 lakh rupees to his company account in February. By June, he had stopped responding to my calls and emails. I realized the company did not exist.',
    'duration_seconds': 180.0
}

analysis = transcriber.analyze_testimony_content(mock_testimony)
print('=== TESTIMONY ANALYSIS ===')
print(f'Credibility Score: {analysis.get("credibility_score", 0):.1f}/10')
print('\nKey Statements:')
for s in analysis.get('key_statements', [])[:3]:
    print(f'  - {s}')
print('\nRed Flags:')
for f in analysis.get('red_flags', [])[:3]:
    print(f'  ! {f}')

## 5. Cross-Witness Inconsistency Detection

In [ ]:
testimonies = [
    {
        'file_name': 'victim_testimony.mp3',
        'transcript': 'I met the accused at a conference in January 2024. He promised 30% returns. I transferred Rs. 45 lakhs in February 2024.',
        'speaker': 'Victim'
    },
    {
        'file_name': 'witness1_testimony.mp3',
        'transcript': 'I saw the accused at the conference in March 2024. He was selling investment plans. He told me he had offices in Mumbai and Delhi.',
        'speaker': 'Witness 1'
    },
    {
        'file_name': 'accused_statement.mp3',
        'transcript': 'I did have a legitimate investment company. All returns depend on market conditions. No promises were made about guaranteed returns. The victim was aware of the risks.',
        'speaker': 'Accused'
    },
]

inconsistencies = transcriber.detect_inconsistencies(testimonies)
print('=== CROSS-WITNESS INCONSISTENCIES ===')
print(f'Total inconsistencies found: {len(inconsistencies.get("inconsistencies", []))}')
for inc in inconsistencies.get('inconsistencies', [])[:3]:
    print(f'\n  Topic: {inc.get("topic", "")}') 
    print(f'  Conflict: {inc.get("conflict", "")}')

## 6. Evidence Matrix — Strength Ranking

In [ ]:
from src.evidence_scorer import EvidenceScorer
import pandas as pd
import matplotlib.pyplot as plt

scorer = EvidenceScorer()

all_evidence = [
    {'file_name': 'bank_statements.pdf',        'doc_type': 'pdf',   'evidence_strength': 9.0, 'category': 'Documentary'},
    {'file_name': 'fir_report.pdf',             'doc_type': 'pdf',   'evidence_strength': 8.5, 'category': 'Documentary'},
    {'file_name': 'fraud_evidence_photo.jpg',   'doc_type': 'image', 'evidence_strength': 7.8, 'category': 'Forensic'},
    {'file_name': 'contract_document.pdf',      'doc_type': 'pdf',   'evidence_strength': 7.5, 'category': 'Documentary'},
    {'file_name': 'victim_testimony.mp3',       'doc_type': 'audio', 'evidence_strength': 7.2, 'category': 'Testimonial'},
    {'file_name': 'email_communications.pdf',   'doc_type': 'pdf',   'evidence_strength': 8.0, 'category': 'Documentary'},
    {'file_name': 'witness_testimony.mp3',      'doc_type': 'audio', 'evidence_strength': 6.5, 'category': 'Testimonial'},
    {'file_name': 'company_registration.pdf',   'doc_type': 'pdf',   'evidence_strength': 6.0, 'category': 'Documentary'},
]

matrix = scorer.build_evidence_matrix(all_evidence)
aggregate = scorer.calculate_aggregate_strength(all_evidence)

df = pd.DataFrame(all_evidence).sort_values('evidence_strength', ascending=False)
print('Evidence Matrix (sorted by strength):')
print(df[['file_name', 'doc_type', 'category', 'evidence_strength']].to_string(index=False))
print(f'\nAggregate Evidence Strength: {aggregate:.2f}/10')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart — strength by file
colors = ['#27ae60' if s >= 8 else '#f39c12' if s >= 7 else '#c0392b' for s in df['evidence_strength']]
axes[0].barh(df['file_name'], df['evidence_strength'], color=colors)
axes[0].axvline(x=7, color='gray', linestyle='--', alpha=0.7, label='Threshold')
axes[0].set_xlim(0, 10)
axes[0].set_xlabel('Strength Score (/10)')
axes[0].set_title('Evidence Strength by Document')
axes[0].legend()

# Pie chart — by category
cat_counts = df.groupby('category')['evidence_strength'].mean()
axes[1].pie(
    cat_counts.values,
    labels=cat_counts.index,
    autopct='%1.0f%%',
    colors=['#3498db', '#e74c3c', '#2ecc71']
)
axes[1].set_title('Avg Strength by Evidence Category')

plt.suptitle('LexAI Evidence Analysis — Case 002', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

The multimodal evidence pipeline:
- **Images**: GPT-4V forensic analysis + EXIF tampering detection
- **Audio**: Whisper transcription + credibility scoring + inconsistency detection
- **Evidence matrix**: Aggregate strength calculation, category breakdown

These evidence scores feed directly into the ML verdict predictor in Notebook 06.